# Core idea: 
### Build a complete, production-style ML pipeline (not just a notebook that gets accuracy once) that takes raw Titanic data in and gives a survival prediction out — using everything you've studied, chained together in one Pipeline object.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [3]:
# Load the built-in Titanic dataset
df = sns.load_dataset('titanic')

## What "done" looks like:

- One clean Pipeline object: imputation → feature engineering (family_size, is_alone) → encoding → scaling/transforms → binning (where useful) → model
- Trained with proper train/test split (no leakage)
- Saved as a .pkl file with joblib
- A GitHub repo with a README explaining your preprocessing decisions (like Q40 from the practice sheet)
- Bonus: a tiny Streamlit app where you type in passenger details and it predicts survival

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


## Preprocessing Plan — Titanic Survival Predictor

| Column | Decision | Notes |
|---|---|---|
| `survived` | Keep as-is | Target variable (y) |
| `pclass` | Ordinal Encoding | Natural rank exists (1st > 2nd > 3rd class status) |
| `class` | Drop | Duplicate info of `pclass`, just as text labels |
| `sex` | One-Hot Encoding | Nominal, no order (Male/Female) |
| `age` | Median Fill → Standardization | Fill missing first, then scale |
| `sibsp` | Drop | Used only to construct `family_size` |
| `parch` | Drop | Used only to construct `family_size` |
| `family_size` | Engineer (`sibsp + parch + 1`) → Standardization | New feature, replaces sibsp/parch |
| `alone` | Drop | Redundant — `family_size` already captures this, with more detail |
| `fare` | Standardization | Continuous, skewed but no fill needed (no NaNs) |
| `embarked` | Mode Fill → One-Hot Encoding | Nominal, few missing values |
| `who` | Drop | Redundant with `sex`/`age` |
| `adult_male` | Drop | Redundant with `sex`/`age` |
| `embark_town` | Drop | Duplicate of `embarked` |
| `alive` | Drop | Direct leak of target (`survived`) — never keep this |
| `deck` | Drop | Too many missing values (~77%) |



yes i used ai for the format but thought about everything myself

In [9]:
# Load data → Engineer family_size → Drop unwanted columns → train_test_split → ColumnTransformer (inside Pipeline)

In [10]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df.drop(columns=['sibsp', 'parch', 'alone', 'class', 'who', 'adult_male', 'deck','embark_town', 'alive'], inplace=True)

In [12]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'fare', 'embarked', 'family_size'], dtype='str')

In [13]:
X = df.drop(columns=['survived'])
y = df['survived']

In [14]:
X

,pclass,sex,age,fare,embarked,family_size
0,3,male,22.0,7.2500,S,2
1,1,female,38.0,71.2833,C,2
2,3,female,26.0,7.9250,S,1
3,1,female,35.0,53.1000,S,2
4,3,male,35.0,8.0500,S,1
...,...,...,...,...,...,...
886,2,male,27.0,13.0000,S,1
887,1,female,19.0,30.0000,S,1
888,3,female,NaN,23.4500,S,4
889,1,male,26.0,30.0000,C,1


In [15]:
y

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: survived, Length: 891, dtype: int64

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
X_train

,pclass,sex,age,fare,embarked,family_size
331,1,male,45.5,28.5000,S,1
733,2,male,23.0,13.0000,S,1
382,3,male,32.0,7.9250,S,1
704,3,male,26.0,7.8542,S,2
813,3,female,6.0,31.2750,S,7
...,...,...,...,...,...,...
106,3,female,21.0,7.6500,S,1
270,1,male,NaN,31.0000,S,1
860,3,male,41.0,14.1083,S,3
435,1,female,14.0,120.0000,S,4


In [18]:
X_test.shape

(179, 6)

In [19]:
y_test.shape

(179,)

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

| Column | Decision | Notes |
|---|---|---|
| `pclass` | Ordinal Encoding | Natural rank exists (1st > 2nd > 3rd class status) |
| `sex` | One-Hot Encoding | Nominal, no order (Male/Female) |
| `age` | Median Fill → Standardization | Fill missing first, then scale |
| `fare` | Standardization | Continuous, skewed but no fill needed (no NaNs) |
| `embarked` | Mode Fill → One-Hot Encoding | Nominal, few missing values |
| `family_size` | Standardization | New feature |

In [31]:
transformers = ColumnTransformer(transformers=[
    ('tnf1', OrdinalEncoder(categories=[[1, 2, 3]]), ['pclass']),
    ('tnf2', OneHotEncoder(drop='first'), ['sex']),
    ('tnf3', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), ['age']),
    ('tnf4', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first'))
    ]), ['embarked']),
    ('tnf5', StandardScaler(), ['fare']),
    ('tnf6', StandardScaler(), ['family_size'])
], remainder='passthrough')

In [32]:
X_train_transformed = transformers.fit_transform(X_train)

In [33]:
X_train_transformed.shape

(712, 7)

In [34]:
from sklearn.linear_model import LogisticRegression

In [35]:
pipe = Pipeline(steps=[
    ('preprocessing', transformers),
    ('model', LogisticRegression())
])

In [36]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['pclass','sex','age','fare','embarked','family_size']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tnf1', ...), ('tnf2', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining c

In [37]:
pipe.score(X_test, y_test)

0.8044692737430168

In [39]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, X, y, cv=5)
scores

array([0.77094972, 0.79213483, 0.79213483, 0.78089888, 0.82022472])

In [40]:
import joblib

joblib.dump(pipe, 'titanic_pipeline.pkl')

['titanic_pipeline.pkl']

In [41]:
loaded_pipe = joblib.load('titanic_pipeline.pkl')
print("Reloaded accuracy:", loaded_pipe.score(X_test, y_test))

Reloaded accuracy: 0.8044692737430168
